# HeatViT-style token pruning on DeiT on Breast Histopathology (IDC) - full pipeline

**Kaggle setup (do this first):**
1. Right panel -> *Add Input* -> search **Breast Histopathology Images** (paultimothymooney) -> Add. (If you skip this, cell 2 downloads it itself - then *Internet* must be **On**.)
2. *Session options* -> Accelerator **GPU T4 x2** (one GPU is used) and Internet **On** (pretrained DeiT weights are downloaded).
3. Run cells top to bottom. For the long run use **Save Version -> Save & Run All (Commit)** so it finishes in the background.

**New:** section 7 reproduces the *comparison* of the paper's Table I / Fig. 2 on this dataset (equal-compute comparison against static, adaptive and head-pruning baselines + a HeatViT sweep) and draws a Fig. 2-style plot. Set `RUN_COMPARISON = False` there to skip it (it adds roughly 2-3 h on a T4; lower `SWEEP_EPOCHS` to shorten).

Everything you need at the end is in one file: `/kaggle/working/bc_export.zip` (weights, biases, other params, test images, labels, results, latency, checkpoints).

In [ ]:
!pip install -q kagglehub
import os, glob, math, time, json, copy, shutil, hashlib, collections
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import timm
from PIL import Image
from concurrent.futures import ProcessPoolExecutor
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# ----- clean old results so nothing is silently resumed from an earlier run -----
for f in ('bc_last.pt', 'bc_heatvit.pt', 'bc_teacher.pt'):
    p = f'/kaggle/working/{f}'
    if os.path.exists(p): os.remove(p)
shutil.rmtree('/kaggle/working/export', ignore_errors=True)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, torch.cuda.get_device_name(0) if dev == 'cuda' else '')

MAX_IMAGES = 80000              # balanced across the two classes (more data = higher accuracy)
TEACHER_EPOCHS = 12             # baseline DeiT fine-tuning epochs
CFG = dict(
    arch='deit_tiny_patch16_224',
    sel_blocks=(3, 6, 9),
    keep=(0.90, 0.80, 0.60),    # cumulative keep ratio after each selector (lower = faster, less accurate)
    epochs=30,                  # HeatViT epochs (increase further if time allows)
    bs=128,
    lr_sel=1e-3, lr_bb=5e-5, wd=0.05,
    lam_distill=0.5, lam_ratio=2.0,
    ramp_frac=0.4,              # keep ratio ramps 1.0 -> target over the first 40% of training
)
amp = dict(device_type='cuda', dtype=torch.float16, enabled=(dev == 'cuda'))

## 2. Data: find images, balance classes, split by patient, cache resized copies

In [ ]:
def find_roots():
    r = []
    for base in ('/kaggle/input', '/kaggle/input/datasets'):
        if os.path.isdir(base):
            for d in os.listdir(base):
                p = f'{base}/{d}'
                if d not in ('notebooks', 'datasets', 'competitions') and os.path.isdir(p):
                    r.append(p)
    r += [p for p in glob.glob('/kaggle/input/datasets/*/*') if os.path.isdir(p)]
    return sorted(set(r))

roots = find_roots()
print('mounted:', roots)

if not roots:                                   # nothing attached -> download the IDC dataset directly
    import kagglehub
    ds_path = kagglehub.dataset_download('paultimothymooney/breast-histopathology-images')
    print('downloaded to:', ds_path)
    roots = [ds_path]

paths = sorted(p for r in roots for p in glob.glob(f'{r}/**/*', recursive=True)
               if p.lower().endswith(('.jpeg', '.jpg', '.png', '.tif', '.tiff')))
assert paths, 'no images found: this dataset is tabular (CSV), HeatViT needs images'
df = pd.DataFrame(dict(path=paths))
df['name'] = df.path.map(os.path.basename)
df = df.drop_duplicates('name').reset_index(drop=True)
df['cls'] = df.path.map(lambda p: os.path.basename(os.path.dirname(p)))
df['grp'] = df.path.map(lambda p: os.path.basename(os.path.dirname(os.path.dirname(p))))
print('total images:', len(df), '| parent folders:', collections.Counter(df.cls).most_common(6))

if {'0', '1'} <= set(df.cls):                   # IDC layout: <patient>/0|1/*.png
    df = df[df.cls.isin(['0', '1'])].reset_index(drop=True)

syn = sorted(df.cls.unique()); cls_idx = {c: i for i, c in enumerate(syn)}
assert 2 <= len(syn) <= 20, f'parent folders do not look like classes: {syn[:10]}'
NUM_CLASSES = len(syn); df['label'] = df.cls.map(cls_idx)
print('classes:', cls_idx, '| images used:', len(df))

per = MAX_IMAGES // NUM_CLASSES
df = pd.concat([g.sample(min(len(g), per), random_state=SEED) for _, g in df.groupby('label')]).reset_index(drop=True)

rs = np.random.RandomState(SEED)
if df.grp.nunique() > 20:                       # split by patient to avoid leakage
    g = df.grp.unique(); rs.shuffle(g); n = len(g)
    gtr, gva = set(g[:int(.7 * n)]), set(g[int(.7 * n):int(.85 * n)])
    split = df.grp.map(lambda x: 'train' if x in gtr else ('val' if x in gva else 'test'))
else:
    split = pd.Series(rs.choice(['train', 'val', 'test'], len(df), p=[.7, .15, .15]), index=df.index)
train_df, val_df, test_df = [df[split == s].reset_index(drop=True) for s in ('train', 'val', 'test')]
assert not (set(train_df.path) & set(test_df.path)) and len(test_df) > 0

CACHE = '/tmp/bc256'; os.makedirs(CACHE, exist_ok=True)
def shrink(p):                                  # JPEG q95 cache keeps disk use small
    out = f'{CACHE}/{hashlib.md5(p.encode()).hexdigest()}.jpg'
    if not os.path.exists(out):
        Image.open(p).convert('RGB').resize((256, 256), Image.BICUBIC).save(out, quality=95)
    return out
for d in (train_df, val_df, test_df):
    d['orig'] = d.path
    with ProcessPoolExecutor(4) as ex: d['path'] = list(ex.map(shrink, d.orig.values, chunksize=64))
for n, d in (('train', train_df), ('val', val_df), ('test', test_df)):
    print(n, len(d), dict(sorted(collections.Counter(d.label).items())))

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
tf_train = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomChoice([T.RandomRotation((a, a)) for a in (0, 90, 180, 270)]),
    T.ColorJitter(0.15, 0.15, 0.10, 0.02),
    T.ToTensor(), T.Normalize(MEAN, STD)])
tf_test = T.Compose([T.Resize(256, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224),
                     T.ToTensor(), T.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, d, tf): self.p, self.y, self.tf = d.path.values, d.label.values, tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB')), int(self.y[i])

def make_loader(d, tf, bs, train):
    return DataLoader(ImgDS(d, tf), batch_size=bs, shuffle=train, num_workers=4,
                      pin_memory=True, drop_last=train, persistent_workers=train)

train_loader = make_loader(train_df, tf_train, CFG['bs'], True)
val_loader = make_loader(val_df, tf_test, 256, False)
test_loader = make_loader(test_df, tf_test, 256, False)

## 3. Model: multi-head token selector + package token (masked for training, `forward_dense` for real pruning)

In [ ]:
class HeadwiseLinear(nn.Module):
    def __init__(self, h, i, o):
        super().__init__()
        self.w = nn.Parameter(torch.randn(h, i, o) * i ** -0.5)
        self.b = nn.Parameter(torch.zeros(h, o))
    def forward(self, x):
        return torch.einsum('bnhi,hio->bnho', x, self.w) + self.b

class TokenSelector(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.h, self.d = heads, dim // heads
        d, h = self.d, heads
        self.local = HeadwiseLinear(h, d, d // 2)
        self.mlp1 = HeadwiseLinear(h, d, d // 2)
        self.mlp2 = HeadwiseLinear(h, d // 2, d // 4)
        self.mlp3 = HeadwiseLinear(h, d // 4, 2)
        self.att1, self.att2 = nn.Linear(h, h), nn.Linear(h, h)
        with torch.no_grad():
            self.mlp3.b[:, 0] = 3.0

    def forward(self, x, policy):
        B, N, D = x.shape
        xh = x.reshape(B, N, self.h, self.d)
        loc = F.gelu(self.local(xh))
        pol = policy[:, :, None, None].to(loc.dtype)
        glob_ = (loc * pol).sum(1, keepdim=True) / (pol.sum(1, keepdim=True) + 1e-6)
        e = torch.cat([loc, glob_.expand(-1, N, -1, -1)], -1)
        s = F.softmax(self.mlp3(F.gelu(self.mlp2(F.gelu(self.mlp1(e))))).float(), -1)
        a = torch.sigmoid(self.att2(F.gelu(self.att1(xh.mean(-1))))).float()
        return (s * a.unsqueeze(-1)).sum(2) / (a.sum(-1, keepdim=True) + 1e-6)

def attn_policy(attn, x, pol):
    B, T_, C = x.shape
    H = attn.num_heads
    q, k, v = attn.qkv(x).reshape(B, T_, 3, H, C // H).permute(2, 0, 3, 1, 4).unbind(0)
    q = getattr(attn, 'q_norm', nn.Identity())(q)
    k = getattr(attn, 'k_norm', nn.Identity())(k)
    a = (q @ k.transpose(-2, -1)).float() * (C // H) ** -0.5
    a = a - a.amax(-1, keepdim=True)
    eye = torch.eye(T_, device=x.device, dtype=a.dtype)[None, None]
    p = pol[:, None, None, :].to(a.dtype)
    p = p + (1 - p) * eye
    e = a.exp() * p
    a = (e / (e.sum(-1, keepdim=True) + 1e-6)).to(v.dtype)
    out = (attn.attn_drop(a) @ v).transpose(1, 2).reshape(B, T_, C)
    return attn.proj_drop(attn.proj(out))

def block_policy(blk, x, pol):
    ident = nn.Identity()
    ls1, ls2 = getattr(blk, 'ls1', ident), getattr(blk, 'ls2', ident)
    dp1 = getattr(blk, 'drop_path1', getattr(blk, 'drop_path', ident))
    dp2 = getattr(blk, 'drop_path2', getattr(blk, 'drop_path', ident))
    x = x + dp1(ls1(attn_policy(blk.attn, blk.norm1(x), pol)))
    return x + dp2(ls2(blk.mlp(blk.norm2(x))))

def count_macs(tokens, D):
    return 12 * tokens * D * D + 2 * tokens * tokens * D

class HeatViT(nn.Module):
    # mode 'mh'    : HeatViT multi-head selector (+ package token if use_package)          -> the paper's method
    # mode 'sh'    : single-head learned selector, normally with use_package=False          -> proxy for adaptive methods (IA-RED2 / Evo-ViT style)
    # mode 'static': fixed, input-agnostic centre-first token order, no parameters          -> proxy for static methods (DynamicViT / ATS / PS-ViT class)
    def __init__(self, backbone, sel_blocks, keep, use_package=True, mode='mh'):
        super().__init__()
        self.bb, self.use_package, self.mode = backbone, use_package, mode
        self.sel_blocks, self.keep = list(sel_blocks), list(keep)
        D, H = backbone.embed_dim, backbone.blocks[0].attn.num_heads
        self.D = D
        self.nc = backbone.num_classes
        if mode == 'static':
            self.selectors = nn.ModuleList()
        else:
            self.selectors = nn.ModuleList([TokenSelector(D, 1 if mode == 'sh' else H) for _ in self.sel_blocks])
        g = int(round(math.sqrt(backbone.patch_embed.num_patches)))
        yy, xx = torch.meshgrid(torch.arange(g), torch.arange(g), indexing='ij')
        dist = ((yy - (g - 1) / 2) ** 2 + (xx - (g - 1) / 2) ** 2).float().sqrt().flatten()
        prior = 1 - dist / dist.max() * 0.999 + torch.arange(g * g).float() * 1e-6   # tie-breaker keeps order deterministic
        self.register_buffer('prior', prior)

    def _score(self, j, xp, policy, pos=None):
        if self.mode == 'static':
            pr = self.prior if pos is None else self.prior[pos]
            pr = pr.unsqueeze(0).expand(xp.shape[0], -1) if pr.dim() == 1 else pr
            return torch.stack([pr, 1 - pr], -1).float()
        return self.selectors[j](xp, policy)

    def forward(self, img):                      # masked version: used for training and accuracy/GMAC counting
        bb = self.bb
        x = bb.patch_embed(img)
        x = torch.cat([bb.cls_token.expand(x.shape[0], -1, -1), x], 1) + bb.pos_embed
        x = bb.pos_drop(x)
        B, N = x.shape[0], x.shape[1] - 1
        policy = x.new_ones(B, N)
        extra, keep_masks, macs = 0, [], 0.
        for i, blk in enumerate(bb.blocks):
            if i in self.sel_blocks:
                j = self.sel_blocks.index(i)
                score = self._score(j, x[:, 1:1 + N], policy)
                p_keep = score[..., 0]
                r = self.keep[j] / (self.keep[j - 1] if j > 0 else 1.0)       # fraction of alive tokens to keep
                k = max(1, int(round(r * policy.sum(1)[0].item())))
                idx = p_keep.masked_fill(policy == 0, -1e4).topk(k, dim=1).indices
                hard_k = torch.zeros_like(policy).scatter_(1, idx, 1.0)
                hard = hard_k + p_keep - p_keep.detach()                       # exact 0/1 forward, gradient to selector
                pruned = policy * (1 - hard)
                if self.use_package:
                    w = (score[..., 0] * pruned).unsqueeze(-1)
                    pkg = (x[:, 1:1 + N].float() * w).sum(1, keepdim=True) / (w.sum(1, keepdim=True) + 1e-6)
                    x = torch.cat([x, pkg.to(x.dtype)], 1)
                    extra += 1
                policy = policy * hard
                keep_masks.append(policy)
            pol_all = torch.cat([policy.new_ones(B, 1), policy, policy.new_ones(B, extra)], 1)
            tokens = 1 + policy.detach().sum(1) + extra
            macs = macs + count_macs(tokens, self.D)
            x = block_policy(blk, x, pol_all)
        macs = macs + N * 768 * self.D + self.D * self.nc
        logits = bb.forward_head(bb.norm(x))
        return logits, dict(keep_masks=keep_masks, macs=macs)

    @torch.no_grad()
    def forward_dense(self, img):                # physically pruned inference: tensors really shrink -> real latency
        bb = self.bb
        x = bb.patch_embed(img)
        x = torch.cat([bb.cls_token.expand(x.shape[0], -1, -1), x], 1) + bb.pos_embed
        x = bb.pos_drop(x)
        B = x.shape[0]
        n_alive = x.shape[1] - 1
        pos = torch.arange(n_alive, device=x.device).unsqueeze(0).expand(B, -1)   # original patch index of every alive token
        for i, blk in enumerate(bb.blocks):
            if i in self.sel_blocks:
                j = self.sel_blocks.index(i)
                xp = x[:, 1:1 + n_alive]
                score = self._score(j, xp, x.new_ones(B, n_alive), pos)
                p_keep = score[..., 0]
                r = self.keep[j] / (self.keep[j - 1] if j > 0 else 1.0)
                k = max(1, int(round(r * n_alive)))
                idx = p_keep.topk(k, dim=1).indices
                kept = xp.gather(1, idx.unsqueeze(-1).expand(-1, -1, xp.shape[-1]))
                old_extra = x[:, 1 + n_alive:]
                if self.use_package:
                    hard_k = torch.zeros_like(p_keep).scatter_(1, idx, 1.0)
                    w = (p_keep * (1 - hard_k)).unsqueeze(-1)
                    pkg = (xp.float() * w).sum(1, keepdim=True) / (w.sum(1, keepdim=True) + 1e-6)
                    x = torch.cat([x[:, :1], kept, old_extra, pkg.to(x.dtype)], 1)
                else:
                    x = torch.cat([x[:, :1], kept, old_extra], 1)
                pos = pos.gather(1, idx)
                n_alive = k
            x = blk(x)
        return bb.forward_head(bb.norm(x))

def baseline_gmacs(D=192, depth=12, tokens=197):
    return (depth * count_macs(tokens, D) + 196 * 768 * D + D * NUM_CLASSES) / 1e9

def heatvit_loss(logits, t_logits, labels, keep_masks, targets, lam_d, lam_r):
    ce = F.cross_entropy(logits.float(), labels)
    kd = F.kl_div(F.log_softmax(logits.float(), -1), F.softmax(t_logits.float(), -1), reduction='batchmean')
    ratio = sum((m.mean() - t) ** 2 for m, t in zip(keep_masks, targets)) if len(keep_masks) else torch.zeros((), device=logits.device)
    return ce + lam_d * kd + lam_r * ratio, ce.detach(), kd.detach(), ratio.detach()

# ---------------- head-pruning baseline (design scheme (3) in the paper's Table I) ----------------
def block_headmask(blk, x, hm):
    ident = nn.Identity()
    ls1, ls2 = getattr(blk, 'ls1', ident), getattr(blk, 'ls2', ident)
    dp1 = getattr(blk, 'drop_path1', getattr(blk, 'drop_path', ident))
    dp2 = getattr(blk, 'drop_path2', getattr(blk, 'drop_path', ident))
    a = blk.attn
    B, N, C = x.shape
    H = a.num_heads
    q, k, v = a.qkv(blk.norm1(x)).reshape(B, N, 3, H, C // H).permute(2, 0, 3, 1, 4).unbind(0)
    q, k = getattr(a, 'q_norm', ident)(q), getattr(a, 'k_norm', ident)(k)
    att = ((q @ k.transpose(-2, -1)) * (C // H) ** -0.5).float().softmax(-1).to(v.dtype)
    o = (a.attn_drop(att) @ v) * hm[None, :, None, None].to(v.dtype)        # pruned heads contribute nothing
    o = o.transpose(1, 2).reshape(B, N, C)
    x = x + dp1(ls1(a.proj_drop(a.proj(o))))
    return x + dp2(ls2(blk.mlp(blk.norm2(x))))

class HeadPruned(nn.Module):
    def __init__(self, backbone, prune_blocks, n_prune=1):
        super().__init__()
        self.bb = backbone
        self.D, self.H, self.nc = backbone.embed_dim, backbone.blocks[0].attn.num_heads, backbone.num_classes
        d = self.D // self.H
        hm = torch.ones(len(backbone.blocks), self.H)
        for i in prune_blocks:                                               # drop the n_prune least important heads (weight magnitude)
            a = backbone.blocks[i].attn
            imp = a.qkv.weight.detach().abs().view(3, self.H, d, self.D).sum((0, 2, 3)) + \
                  a.proj.weight.detach().abs().view(self.D, self.H, d).sum((0, 2))
            hm[i, imp.argsort()[:n_prune]] = 0
        self.register_buffer('hm', hm)
        self.keep = []

    def forward(self, img):
        bb = self.bb
        x = bb.patch_embed(img)
        x = torch.cat([bb.cls_token.expand(x.shape[0], -1, -1), x], 1) + bb.pos_embed
        x = bb.pos_drop(x)
        for i, blk in enumerate(bb.blocks):
            x = block_headmask(blk, x, self.hm[i])
        logits = bb.forward_head(bb.norm(x))
        N, d = x.shape[1], self.D // self.H
        macs = sum(4 * N * self.D * (h * d) + 2 * N * N * (h * d) + 8 * N * self.D * self.D for h in self.hm.sum(1).tolist())
        macs += 196 * 768 * self.D + self.D * self.nc
        return logits, dict(keep_masks=[], macs=torch.full((img.shape[0],), float(macs), device=img.device))


## 4. Evaluation helper + teacher (baseline DeiT) fine-tuning

In [ ]:
@torch.no_grad()
def evaluate(m, loader, pruned=True):
    m.eval()
    correct = n = 0
    macs, keeps = [], []
    for x, y in loader:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        with torch.autocast(**amp):
            if pruned:
                logits, info = m(x)
                macs.append(info['macs'].float().cpu())
                keeps.append(torch.stack([k.mean() for k in info['keep_masks']]).float().cpu() if info['keep_masks'] else torch.zeros(0))
            else:
                logits = m(x)
        correct += (logits.argmax(-1) == y).sum().item(); n += y.numel()
    out = dict(acc=100 * correct / n)
    if pruned:
        out['gmacs'] = torch.cat(macs).mean().item() / 1e9
        out['keep'] = torch.stack(keeps).mean(0).tolist()
    return out

teacher = timm.create_model(CFG['arch'], pretrained=True, num_classes=NUM_CLASSES).to(dev)
opt_t = torch.optim.AdamW(teacher.parameters(), lr=2e-4, weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(opt_t, max_lr=2e-4, total_steps=TEACHER_EPOCHS * len(train_loader))
scaler_t = torch.amp.GradScaler('cuda', enabled=(dev == 'cuda'))
best_t = 0.
for ep in range(TEACHER_EPOCHS):
    teacher.train(); t0 = time.time()
    for x, y in train_loader:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        with torch.autocast(**amp):
            loss = F.cross_entropy(teacher(x).float(), y, label_smoothing=0.1)
        opt_t.zero_grad(set_to_none=True)
        scaler_t.scale(loss).backward(); scaler_t.step(opt_t); scaler_t.update(); sched.step()
    r = evaluate(teacher, val_loader, pruned=False)
    print(f"teacher epoch {ep + 1}/{TEACHER_EPOCHS}  val acc {r['acc']:.2f}%  ({time.time() - t0:.0f}s)")
    if r['acc'] >= best_t:
        best_t = r['acc']; torch.save(teacher.state_dict(), '/kaggle/working/bc_teacher.pt')

teacher.load_state_dict(torch.load('/kaggle/working/bc_teacher.pt', map_location=dev))
teacher.eval()
for p in teacher.parameters(): p.requires_grad = False
BASE_G = baseline_gmacs(D=teacher.embed_dim, depth=len(teacher.blocks))

student_bb = timm.create_model(CFG['arch'], pretrained=False, num_classes=NUM_CLASSES)
student_bb.load_state_dict(teacher.state_dict())
model = HeatViT(student_bb, CFG['sel_blocks'], CFG['keep']).to(dev)

base = evaluate(teacher, test_loader, pruned=False)
print(f"Baseline DeiT (test)  acc = {base['acc']:.2f}%   GMACs = {BASE_G:.2f}")
print("Pruned model BEFORE training (test):", evaluate(model, test_loader))

## 5. Train HeatViT (progressive keep-ratio ramp, distillation from the teacher)

In [ ]:
opt = torch.optim.AdamW([
    dict(params=model.selectors.parameters(), lr=CFG['lr_sel']),
    dict(params=model.bb.parameters(), lr=CFG['lr_bb'])], weight_decay=CFG['wd'])
steps = CFG['epochs'] * len(train_loader)
ramp_steps = max(1, int(CFG['ramp_frac'] * steps))
ramp_epochs = math.ceil(CFG['ramp_frac'] * CFG['epochs'])
base_lrs = [g['lr'] for g in opt.param_groups]
scaler = torch.amp.GradScaler('cuda', enabled=(dev == 'cuda'))
FINAL_KEEP = list(CFG['keep'])

CKPT, BEST = '/kaggle/working/bc_last.pt', '/kaggle/working/bc_heatvit.pt'
start_ep, step, history, best_acc = 0, 0, [], 0.
if os.path.exists(CKPT):                         # resume if the session was interrupted
    ck = torch.load(CKPT, map_location=dev)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); scaler.load_state_dict(ck['scaler'])
    start_ep, step, history, best_acc = ck['ep'], ck['step'], ck['history'], ck['best']
    print('resumed from epoch', start_ep)

for ep in range(start_ep, CFG['epochs']):
    model.train(); t0 = time.time(); tot = 0.
    for x, y in train_loader:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        prog = min(1., step / ramp_steps)
        model.keep = [1 - (1 - k) * prog for k in FINAL_KEEP]
        lr_scale = min(1., (step + 1) / 100) * 0.5 * (1 + math.cos(math.pi * step / steps))
        for g, b in zip(opt.param_groups, base_lrs): g['lr'] = b * lr_scale
        with torch.autocast(**amp):
            with torch.no_grad(): t_logits = teacher(x)
            logits, info = model(x)
            loss, ce, kd, ratio = heatvit_loss(logits, t_logits, y, info['keep_masks'], model.keep,
                                               CFG['lam_distill'], CFG['lam_ratio'])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item(); step += 1
    cur_keep = model.keep
    model.keep = FINAL_KEEP                      # validate at the real target compute
    r = evaluate(model, val_loader)
    model.keep = cur_keep
    history.append(dict(epoch=ep + 1, loss=tot / len(train_loader), **r))
    if r['acc'] > best_acc and (ep + 1) >= ramp_epochs:
        best_acc = r['acc']; torch.save(model.state_dict(), BEST)
    print(f"epoch {ep + 1}/{CFG['epochs']}  loss {tot / len(train_loader):.3f}  "
          f"| val@target acc {r['acc']:.2f}%  GMACs {r['gmacs']:.2f}  keep {[round(k, 2) for k in r['keep']]}  "
          f"({time.time() - t0:.0f}s)")
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                    ep=ep + 1, step=step, history=history, best=best_acc), CKPT)
model.keep = FINAL_KEEP

## 6. Final test accuracy (+ flip TTA), measured latency (GPU and CPU) and ZCU102 latency estimate

In [ ]:
model.load_state_dict(torch.load(BEST, map_location=dev))
model.keep = list(CFG['keep']); model.eval(); teacher.eval()

@torch.no_grad()
def evaluate_tta(m, loader):
    c = n = 0
    for x, y in loader:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        with torch.autocast(**amp):
            p = 0
            for v in (x, x.flip(3), x.flip(2)):
                o = m(v); o = o[0] if isinstance(o, tuple) else o
                p = p + F.softmax(o.float(), -1)
        c += (p.argmax(-1) == y).sum().item(); n += y.numel()
    return 100 * c / n

base = evaluate(teacher, test_loader, pruned=False)
final = evaluate(model, test_loader)
base_tta, final_tta = evaluate_tta(teacher, test_loader), evaluate_tta(model, test_loader)

xb = next(iter(test_loader))[0][:16].to(dev)    # sanity: really-pruned path must match the masked path
with torch.no_grad():
    d = (model(xb)[0].float() - model.forward_dense(xb).float()).abs().max().item()
print(f'max |logit difference| masked vs dense forward (fp32): {d:.2e}  (should be ~1e-4 or smaller)')

def _time(fn, x, device, warmup, iters):
    for _ in range(warmup): fn(x)
    if device == 'cuda':
        torch.cuda.synchronize(); ts = []
        for _ in range(iters):
            s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            s.record(); fn(x); e.record(); torch.cuda.synchronize(); ts.append(s.elapsed_time(e))
    else:
        ts = []
        for _ in range(iters):
            t0 = time.perf_counter(); fn(x); ts.append((time.perf_counter() - t0) * 1e3)
    return np.array(ts)

def make_fn(f, device):
    @torch.no_grad()
    def g(x):
        if device == 'cuda':
            with torch.autocast('cuda', dtype=torch.float16): return f(x)
        return f(x)
    return g

rows = []
def bench(name, f, device, batches, warmup, iters):
    fn = make_fn(f, device)
    for b in batches:
        x = torch.randn(b, 3, 224, 224, device=device)
        ts = _time(fn, x, device, warmup, iters)
        rows.append(dict(model=name, device=device, batch=b, mean_ms=ts.mean(), p50_ms=np.median(ts),
                         p95_ms=np.percentile(ts, 95), ms_per_img=ts.mean() / b, fps=1e3 * b / ts.mean()))

if dev == 'cuda':
    bench('Baseline DeiT', teacher, 'cuda', (1, 8, 64), 30, 150)
    bench('HeatViT (pruned, dense)', model.forward_dense, 'cuda', (1, 8, 64), 30, 150)
torch.set_num_threads(4)
cpu_t = copy.deepcopy(teacher).cpu().float().eval()
cpu_m = copy.deepcopy(model).cpu().float().eval()
bench('Baseline DeiT', cpu_t, 'cpu', (1, 8), 3, 20)
bench('HeatViT (pruned, dense)', cpu_m.forward_dense, 'cpu', (1, 8), 3, 20)
del cpu_t, cpu_m
lat = pd.DataFrame(rows).round(3)
print(lat.to_string(index=False))
sp = lat.pivot_table(index=['device', 'batch'], columns='model', values='mean_ms')
sp['speedup'] = sp['Baseline DeiT'] / sp['HeatViT (pruned, dense)']
print('\nMeasured speed-up (baseline ms / pruned ms):'); print(sp.round(3).to_string())

# ZCU102 latency ESTIMATE from the paper's measured per-block latencies (Table IV; transformer blocks only, 16-bit)
_KEEP = np.array([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
_TAB = {192: np.array([0.636, 0.702, 0.764, 0.881, 0.945, 1.034]),      # DeiT-T
        384: np.array([1.682, 1.973, 2.255, 2.565, 2.837, 3.161])}      # DeiT-S
_LAT = _TAB[model.D]; _FIT = np.polyfit(_KEEP, _LAT, 1)
def block_ms(r): return float(np.interp(r, _KEEP, _LAT)) if r >= 0.5 else float(np.polyval(_FIT, r))
def fpga_estimate_ms(keep, sel_blocks, depth, use_package=True):
    n, extra, tot = 196, 0, 0.
    for i in range(depth):
        if i in sel_blocks:
            j = list(sel_blocks).index(i)
            n = max(1, int(round(keep[j] / (keep[j - 1] if j > 0 else 1.0) * n))); extra += int(use_package)
        tot += block_ms((1 + n + extra) / 197)
    return tot
fp_base = len(model.bb.blocks) * block_ms(1.0)
fp_pruned = fpga_estimate_ms(CFG['keep'], CFG['sel_blocks'], len(model.bb.blocks), model.use_package)
print(f'\nESTIMATED ZCU102 latency (paper Table IV, blocks only, 16-bit): baseline {fp_base:.2f} ms -> '
      f'HeatViT {fp_pruned:.2f} ms ({fp_base / fp_pruned:.2f}x); with the paper\'s ~1.9x 8-bit factor: {fp_pruned / 1.9:.2f} ms')

print('\n=== Summary (held-out test set) ===')
print(f"Baseline DeiT   : acc {base['acc']:.2f}% (flip-TTA {base_tta:.2f}%)  GMACs {BASE_G:.2f}")
print(f"HeatViT (ours)  : acc {final['acc']:.2f}% (flip-TTA {final_tta:.2f}%)  GMACs {final['gmacs']:.2f}  "
      f"({100 * (1 - final['gmacs'] / BASE_G):.1f}% fewer)  keep/stage {[round(k, 2) for k in final['keep']]}")
RESULTS = dict(cfg={k: list(v) if isinstance(v, tuple) else v for k, v in CFG.items()},
               classes=cls_idx, baseline=base, final=final, baseline_tta=base_tta, final_tta=final_tta,
               baseline_gmacs=BASE_G, latency_measured=lat.to_dict('records'),
               fpga_estimate_ms=dict(baseline_16bit=fp_base, heatvit_16bit=fp_pruned, heatvit_8bit_est=fp_pruned / 1.9),
               history=history)
json.dump(RESULTS, open('/kaggle/working/bc_results.json', 'w'), indent=1)

## 7. Comparison like the paper's Table I / Fig. 2 (equal compute, same data, same training budget)
The paper's Fig. 2 compares HeatViT with DyViT, IA, VTP, S2ViTE, ATS, PS-ViT, Evo-ViT and UP-DeiT **on ImageNet-1K**. Those exact numbers need ImageNet-1K, each paper's official code and many GPU-days, so they cannot be re-run on Kaggle.
Instead this section re-implements one *representative proxy per design scheme of Table I* inside this model and compares them with HeatViT on **your dataset** at the same GMACs:

| Table I scheme | Papers | Proxy trained here |
|---|---|---|
| (1) static token pruning | DyViT, ATS, PS-ViT | fixed, input-agnostic keep order (centre patches first), no package token |
| (2) adaptive token pruning | IA, Evo-ViT | learned single-head selector, no package token |
| (3) head pruning | VTP, S2ViTE, UP-DeiT | magnitude head pruning, same fine-tuning |
| HeatViT | this paper | multi-head selector + package token, keep-ratio sweep |

These are simplified stand-ins, not the original methods; say so in the report. Every row is trained for `SWEEP_EPOCHS` with the same distillation recipe.

In [ ]:
RUN_COMPARISON = False
SWEEP_EPOCHS = 10                                   # same budget for every method; raise for better numbers
COMPARE_KEEP = (0.90, 0.80, 0.60)                   # keep ratios used by the static / adaptive proxies
SWEEP_KEEPS = [(0.95, 0.90, 0.80), (0.90, 0.80, 0.60), (0.80, 0.60, 0.40), (0.70, 0.39, 0.21)]   # HeatViT sweep (last = paper's DeiT-T setting)
HEAD_PRUNE_BLOCKS = list(range(4, len(teacher.blocks)))  # prune 1 head in each of these blocks

# ---- Table I (as in the paper) and the paper's reported Fig. 2 numbers (ImageNet-1K) for the report ----
TABLE_I = pd.DataFrame([
    ('DyViT [41]', '1 static token pruning'), ('ATS [13]', '1 static token pruning'),
    ('IA [36]', '2 adaptive token pruning'), ('PS-ViT [44]', '1 static token pruning'),
    ('VTP [64]', '3 head pruning'), ('Evo-ViT [53]', '2 adaptive token pruning'),
    ('S2ViTE [10]', '1+3+4 static token + head + token-channel pruning'), ('UP-DeiT [55]', '3+4 head + token-channel pruning')],
    columns=['method', 'design_scheme'])
PAPER_FIG2 = pd.DataFrame([
    ('HeatViT-T0', 0.54, 70.80), ('HeatViT-T1', 0.90, 72.10), ('HeatViT-T2', 1.00, 72.20), ('HeatViT-T3', 1.29, 76.87),
    ('HeatViT-T4', 1.30, 77.02), ('HeatViT-S0', 1.80, 78.50), ('HeatViT-S1', 2.01, 79.00), ('HeatViT-S2', 2.64, 79.34),
    ('HeatViT-S3', 3.86, 79.80), ('HeatViT-B0', 6.10, 80.10), ('HeatViT-B1', 10.49, 81.05), ('HeatViT-LV-S0', 3.10, 83.05),
    ('HeatViT-LV-S1', 4.28, 83.10), ('HeatViT-LV-M0', 5.30, 82.70), ('HeatViT-LV-M1', 7.32, 83.71)],
    columns=['model', 'gmacs', 'top1_acc_imagenet1k_reported_in_paper'])
TABLE_I.to_csv('/kaggle/working/paper_table1.csv', index=False)
PAPER_FIG2.to_csv('/kaggle/working/paper_fig2_reported.csv', index=False)

def train_pruned(m, epochs, final_keep=None, ramp_frac=0.4, tag='run'):
    sel_params = list(m.selectors.parameters()) if hasattr(m, 'selectors') else []
    groups = [dict(params=m.bb.parameters(), lr=CFG['lr_bb'])]
    if sel_params: groups.insert(0, dict(params=sel_params, lr=CFG['lr_sel']))
    o = torch.optim.AdamW(groups, weight_decay=CFG['wd'])
    st_total = epochs * len(train_loader); ramp_steps = max(1, int(ramp_frac * st_total))
    ramp_ep = math.ceil(ramp_frac * epochs) if final_keep is not None else 0
    blr = [g['lr'] for g in o.param_groups]
    sc = torch.amp.GradScaler('cuda', enabled=(dev == 'cuda'))
    best, st, path = 0., 0, f'/kaggle/working/cmp_{tag}.pt'
    for ep in range(epochs):
        m.train(); t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            if final_keep is not None:
                prog = min(1., st / ramp_steps); m.keep = [1 - (1 - k) * prog for k in final_keep]
            ls = min(1., (st + 1) / 100) * 0.5 * (1 + math.cos(math.pi * st / st_total))
            for g, b in zip(o.param_groups, blr): g['lr'] = b * ls
            with torch.autocast(**amp):
                with torch.no_grad(): tl = teacher(x)
                lg, info = m(x)
                loss, _, _, _ = heatvit_loss(lg, tl, y, info['keep_masks'], getattr(m, 'keep', []),
                                             CFG['lam_distill'], CFG['lam_ratio'])
            o.zero_grad(set_to_none=True)
            sc.scale(loss).backward(); sc.step(o); sc.update(); st += 1
        cur = None
        if final_keep is not None: cur = m.keep; m.keep = list(final_keep)
        r = evaluate(m, val_loader)
        if cur is not None: m.keep = cur
        if r['acc'] > best and (ep + 1) >= ramp_ep:
            best = r['acc']; torch.save(m.state_dict(), path)
        print(f"  [{tag}] epoch {ep + 1}/{epochs}  val acc {r['acc']:.2f}%  GMACs {r['gmacs']:.2f}  ({time.time() - t0:.0f}s)")
    m.load_state_dict(torch.load(path, map_location=dev))
    if final_keep is not None: m.keep = list(final_keep)
    return m

def fresh_bb():
    bb = timm.create_model(CFG['arch'], pretrained=False, num_classes=NUM_CLASSES)
    bb.load_state_dict(teacher.state_dict()); return bb

comp_rows = [dict(method='DeiT baseline', kind='baseline', scheme='unpruned', gmacs=BASE_G, acc=base['acc'], keep='')]
if RUN_COMPARISON:
    def run_one(name, kind, scheme, build, final_keep):
        print(f'\n=== {name} | {scheme} ===')
        m = build().to(dev)
        m = train_pruned(m, SWEEP_EPOCHS, final_keep, tag=name.replace(' ', '_'))
        r = evaluate(m, test_loader)
        comp_rows.append(dict(method=name, kind=kind, scheme=scheme, gmacs=r['gmacs'], acc=r['acc'],
                              keep=' / '.join(f'{k:.2f}' for k in r['keep'])))
        print(f"  TEST: acc {r['acc']:.2f}%  GMACs {r['gmacs']:.2f}")
        del m; torch.cuda.empty_cache()

    run_one('Static (centre-first)', 'static', '(1) static token pruning proxy',
            lambda: HeatViT(fresh_bb(), CFG['sel_blocks'], COMPARE_KEEP, use_package=False, mode='static'), COMPARE_KEEP)
    run_one('Adaptive 1-head', 'adaptive', '(2) adaptive token pruning proxy',
            lambda: HeatViT(fresh_bb(), CFG['sel_blocks'], COMPARE_KEEP, use_package=False, mode='sh'), COMPARE_KEEP)
    run_one('Head pruning', 'head', '(3) head pruning proxy',
            lambda: HeadPruned(fresh_bb(), HEAD_PRUNE_BLOCKS, 1), None)
    for i, kp in enumerate(SWEEP_KEEPS):
        run_one(f'HeatViT-K{i}', 'heatvit', 'multi-head selector + package token',
                lambda kp=kp: HeatViT(fresh_bb(), CFG['sel_blocks'], kp, use_package=True, mode='mh'), kp)

comp_rows.append(dict(method=f"HeatViT main ({CFG['epochs']} ep)", kind='heatvit_main', scheme='main run of this notebook',
                      gmacs=final['gmacs'], acc=final['acc'], keep=' / '.join(f'{k:.2f}' for k in final['keep'])))
cdf = pd.DataFrame(comp_rows)
cdf['gmacs_reduction_%'] = 100 * (1 - cdf.gmacs / BASE_G)
cdf['acc_change_pts'] = cdf.acc - base['acc']
cdf = cdf.round(3)
cdf.to_csv('/kaggle/working/comparison_table.csv', index=False)
print('\n=== Comparison on the held-out test set (equal data, same training budget except the main row) ===')
print(cdf[['method', 'scheme', 'keep', 'gmacs', 'gmacs_reduction_%', 'acc', 'acc_change_pts']].to_string(index=False))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9.5, 6))
sty = {'baseline': ('tab:blue', 'o'), 'static': ('tab:green', 's'), 'adaptive': ('tab:orange', '^'), 'head': ('tab:cyan', 'D')}
for _, r in cdf[cdf.kind.isin(list(sty))].iterrows():
    c, mk = sty[r.kind]; ax.scatter(r.gmacs, r.acc, c=c, marker=mk, s=70, zorder=4)
    ax.annotate(r.method, (r.gmacs, r.acc), textcoords='offset points', xytext=(6, 5), fontsize=9)
hv = cdf[cdf.kind == 'heatvit'].sort_values('gmacs')
if len(hv):
    ax.plot(hv.gmacs, hv.acc, '-', c='red', lw=2, zorder=3); ax.scatter(hv.gmacs, hv.acc, marker='*', s=260, c='red', zorder=5)
    for _, r in hv.iterrows(): ax.annotate(r.method, (r.gmacs, r.acc), c='red', textcoords='offset points', xytext=(6, -14), fontsize=9)
for _, r in cdf[cdf.kind == 'heatvit_main'].iterrows():
    ax.scatter(r.gmacs, r.acc, marker='*', s=300, c='gold', edgecolors='k', zorder=6)
    ax.annotate(r.method, (r.gmacs, r.acc), textcoords='offset points', xytext=(6, 8), fontsize=9)
ax.set_xlabel('GMACs'); ax.set_ylabel('Test Top-1 accuracy (%)'); ax.grid(alpha=.3)
ax.set_title('Accuracy vs computation (Fig. 2 style) - breast histopathology, DeiT backbone')
plt.tight_layout(); plt.savefig('/kaggle/working/comparison_fig2_style.png', dpi=160); plt.show()

RESULTS['comparison'] = cdf.to_dict('records')
json.dump(RESULTS, open('/kaggle/working/bc_results.json', 'w'), indent=1)

## 7. Export everything to ONE zip: weights, biases, other parameters, test images + labels, results, latency, checkpoints

In [ ]:
EXP = '/kaggle/working/export'
shutil.rmtree(EXP, ignore_errors=True)
for sub in ('weights', 'biases', 'other', 'checkpoints'):
    os.makedirs(f'{EXP}/{sub}', exist_ok=True)

# ---- weights / biases / other parameters (float32 .npy, one file per tensor, plus one combined .npz)
model.eval().cpu()
rows_p, all_params = [], {}
for name, p in model.named_parameters():
    arr = p.detach().float().numpy()
    leaf = name.split('.')[-1]
    kind = ('biases' if leaf in ('bias', 'b') else 'weights' if leaf in ('weight', 'w') else 'other')
    np.save(f'{EXP}/{kind}/{name}.npy', arr)
    all_params[name] = arr
    rows_p.append(dict(name=name, kind=kind, shape=str(list(arr.shape)), params=arr.size))
np.savez(f'{EXP}/all_params.npz', **all_params)
summary = pd.DataFrame(rows_p); summary.to_csv(f'{EXP}/param_summary.csv', index=False)
print(summary.groupby('kind').params.agg(['count', 'sum']))
model.to(dev)

# ---- 100 class-balanced test images (224x224 PNG), normalized tensor array, labels + model predictions
IMG_DIR = f'{EXP}/test_images'; os.makedirs(IMG_DIR, exist_ok=True)
n_each = [100 // NUM_CLASSES + (1 if i < 100 % NUM_CLASSES else 0) for i in range(NUM_CLASSES)]
picked = pd.concat([test_df[test_df.label == c].sample(min(n, (test_df.label == c).sum()), random_state=123)
                    for c, n in enumerate(n_each)]).sample(frac=1, random_state=123)
resize_crop = T.Compose([T.Resize(256, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224)])
tensors, meta = [], []
for i, (_, r) in enumerate(picked.iterrows()):
    img = Image.open(r.path).convert('RGB'); fname = f'img_{i:03d}.png'
    resize_crop(img).save(f'{IMG_DIR}/{fname}'); tensors.append(tf_test(img))
    meta.append(dict(file=fname, orig=os.path.basename(r.orig), label=int(r.label), class_name=syn[r.label]))
batch = torch.stack(tensors); np.save(f'{EXP}/test_images_normalized.npy', batch.numpy())
model.eval().to(dev); model.keep = list(CFG['keep'])
with torch.no_grad(): logits, _ = model(batch.to(dev))
pred = logits.argmax(-1).cpu().numpy(); np.save(f'{EXP}/test_logits.npy', logits.float().cpu().numpy())
for m, p_ in zip(meta, pred): m.update(pred=int(p_), pred_name=syn[int(p_)], correct=bool(p_ == m['label']))
labels_df = pd.DataFrame(meta); labels_df.to_csv(f'{EXP}/test_labels.csv', index=False)
print(labels_df.head(8).to_string())
print('class counts:', labels_df.label.value_counts().to_dict(), '| accuracy on these 100:', labels_df.correct.mean() * 100, '%')

# ---- results, latency, checkpoints, README
shutil.copy('/kaggle/working/bc_results.json', f'{EXP}/results.json')
lat.to_csv(f'{EXP}/latency_measured.csv', index=False)
for f in ('comparison_table.csv', 'comparison_fig2_style.png', 'paper_table1.csv', 'paper_fig2_reported.csv'):
    if os.path.exists(f'/kaggle/working/{f}'): shutil.copy(f'/kaggle/working/{f}', f'{EXP}/{f}')
shutil.copy(BEST, f'{EXP}/checkpoints/heatvit_best.pt')
shutil.copy('/kaggle/working/bc_teacher.pt', f'{EXP}/checkpoints/baseline_teacher.pt')
open(f'{EXP}/README.txt', 'w').write(
    'weights/, biases/, other/ : one .npy per parameter tensor (float32); all_params.npz has all of them\n'
    'param_summary.csv        : name, kind, shape, #params for every tensor\n'
    'test_images/             : 100 class-balanced test images (224x224); test_labels.csv has true label + prediction\n'
    'test_images_normalized.npy / test_logits.npy : model input tensors and output logits for those images\n'
    'results.json, latency_measured.csv : accuracy, GMACs, measured latency, ZCU102 latency estimate\n'
    'checkpoints/             : PyTorch state_dicts (HeatViT and baseline)\n'
    'comparison_table.csv, comparison_fig2_style.png : Table I / Fig. 2 style comparison on this dataset\n'
    'paper_table1.csv, paper_fig2_reported.csv       : Table I and the ImageNet-1K numbers REPORTED in the paper (not re-run)\n'
    f'classes: {cls_idx}\n')

zip_path = shutil.make_archive('/kaggle/working/bc_export', 'zip', EXP)
print('\nZIP READY:', zip_path, f'({os.path.getsize(zip_path) / 1e6:.1f} MB)')
from IPython.display import FileLink
FileLink('bc_export.zip')

: 